# Conversion dataframes to datasets

In [ ]:
# Libraries
import os, shutil
import dask.dataframe as dd
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'

---

In [ ]:
# dataset with orig lat and lon for reindexing
ds_worldclim = xr.open_zarr(dir01 + 'ds_prep_worldclim_r001.zarr')

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

In [ ]:
def df_rfpred2ds(var_tar, prim):

    """
    Calculate soc mean, min and max values of predictions from 10 best 
    performing models and export as zarr file
    """
    
    # Transform to dataset and export as intermediate file
    def df2ds(df, i, file_interm):
        """Transform each variable of dataframe to variable of dataset,
        ensure correct lat and lon (reindex_like) and export as intermediate file
        """
        
        # Tranfsform and export
        df[['lat', 'lon', i]] \
            .compute() \
            .set_index(['lat', 'lon']) \
            .to_xarray() \
            .chunk(dict(lat=5000, lon=5000)) \
            .sortby('lat', ascending=False) \
            .fillna(-32768) \
            .round(0) \
            .astype('int16') \
            .reindex_like(ds_worldclim, method=None, fill_value=-32768) \
            .chunk(dict(lat=5000, lon=5000)) \
            .to_zarr(os.path.join(dir03, 'files_predicted_r001', file_interm), 
                     mode='w')

        
    # Name of input file 
    file_in = f'df_rfpred_{ctype}_{etype}.parquet'
    # Get rf data
    file_rfpred = os.path.join(dir03, 'files_predicted_r001', file_in)
    df_rfpred  = dd.read_parquet(file_rfpred)
    
    # Calculate mean values for each grid cell from rf predicition
    rfr_cols = ['rfr_' + str(i) for i in range(1, 11)]
    df_rfpred = df_rfpred.assign(rfr_mean = df_rfpred[rfr_cols].mean(axis=1))

    # Name of interm file
    file_interm = f'ds_interm_{ctype}_{etype}.zarr'

    # Name of output file
    file_out = f'ds_rfpred_{ctype}_{etype}.zarr'
    
    # Transform Random Forest predictions
    df2ds(df_rfpred, 'rfr_mean', file_interm)
    
    # Import datasets, merge and export as zarr
    xr.open_zarr(
        os.path.join(dir03, 'files_predicted_r001', file_interm)) \
        .rename(rfr_mean = f'{ctype}_{etype}') \
        .to_zarr(os.path.join(dir03, 'files_predicted_r001', file_out), 
                 mode='w')
    
    # Delete intermediate datasets
    shutil.rmtree(os.path.join(dir03, 'files_predicted_r001', file_interm))

In [ ]:
# Calculate cveg mean of predictions from 10 best performing models
# Export as zarr file
for etype in list_etype:
    ctype = 'cveg'
    print(f'{ctype}_{etype}')
    %time df_rfpred2ds(ctype, etype)

In [ ]:
## Calculate csoil mean of predictions from 10 best performing models
## Export as zarr file
for etype in list_etype:
    ctype = 'csoil'
    print(f'{ctype}_{etype}')
    %time df_rfpred2ds(ctype, etype)

---

### Post-processing

In [ ]:
# Merge single dataset files to one dataset
for ctype in ['cveg', 'csoil']:
    xr.open_mfdataset(
        os.path.join(dir03, 'files_predicted_r001', f'ds_rfpred_{ctype}_*.zarr'), 
        engine='zarr') \
        .to_zarr(os.path.join(dir03, f'ds_rfpred_{ctype}_r001.zarr'), mode='w');